# 02 — Model Training

**Project:** AuraVision — Crowd Age Group Majority Classifier  
**Purpose:** Train the EfficientNet-B0 transfer-learning model on the split dataset.

All model logic lives in `src/train_model.py`.  
This notebook calls those functions and displays training progress.

**Output:** `models/age_classifier.pth`

## Step 0 — Setup

In [1]:
!pip install -q torch torchvision pillow numpy

import os, sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/AuraVision'
except ImportError:
    PROJECT_ROOT = os.path.abspath('..')

os.chdir(PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
print('Working directory:', os.getcwd())

Working directory: c:\Users\shahd\OneDrive\Desktop\AuraVision



[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 1 — Load Training Data

In [2]:
from train_model import build_dataloaders, TRAIN_DIR, DEVICE

print(f'Device: {DEVICE}\n')
train_loader, val_loader, class_to_idx = build_dataloaders(TRAIN_DIR)

Device: cpu

Classes detected:     ['Adults', 'Children', 'Seniors']
Class-to-index map:   {'Adults': 0, 'Children': 1, 'Seniors': 2}
Train class dist: {'Adults': 221, 'Children': 161, 'Seniors': 135}


## Step 2 — Build Model

EfficientNet-B0 pretrained on ImageNet.  
Backbone layers 0–5 are frozen; layers 6+ and the custom classifier head are trainable.

In [3]:
from train_model import build_model

model = build_model()

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total:,}')
print(f'Trainable params: {trainable:,}  ({100*trainable/total:.1f}%)')

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\shahd/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100.0%


Total params:     4,171,903
Trainable params: 3,320,095  (79.6%)


## Step 3 — Train

Trains for 40 epochs with AdamW + cosine annealing.  
Best checkpoint (by validation accuracy) is saved to `models/age_classifier.pth`.

In [4]:
from train_model import train_model, NUM_EPOCHS

print(f'Training for {NUM_EPOCHS} epochs...\n')
model = train_model(model, train_loader, val_loader)

Training for 40 epochs...

Epoch [01/40]  Loss: 1.0762  Train: 41.97%  Val: 67.03%
Epoch [02/40]  Loss: 0.9944  Train: 57.83%  Val: 78.02%
Epoch [03/40]  Loss: 0.8096  Train: 71.95%  Val: 78.02%
Epoch [04/40]  Loss: 0.5813  Train: 79.11%  Val: 81.32%
Epoch [05/40]  Loss: 0.4950  Train: 81.43%  Val: 79.12%
Epoch [06/40]  Loss: 0.4867  Train: 81.04%  Val: 80.22%
Epoch [07/40]  Loss: 0.3770  Train: 87.62%  Val: 76.92%
Epoch [08/40]  Loss: 0.3733  Train: 86.27%  Val: 82.42%
Epoch [09/40]  Loss: 0.3454  Train: 87.23%  Val: 80.22%
Epoch [10/40]  Loss: 0.2794  Train: 90.33%  Val: 86.81%
Epoch [11/40]  Loss: 0.2843  Train: 89.56%  Val: 79.12%
Epoch [12/40]  Loss: 0.2437  Train: 90.72%  Val: 84.62%
Epoch [13/40]  Loss: 0.2572  Train: 91.30%  Val: 83.52%
Epoch [14/40]  Loss: 0.3108  Train: 87.81%  Val: 82.42%
Epoch [15/40]  Loss: 0.2828  Train: 89.56%  Val: 83.52%
Epoch [16/40]  Loss: 0.2374  Train: 92.26%  Val: 84.62%
Epoch [17/40]  Loss: 0.2212  Train: 91.68%  Val: 79.12%
Epoch [18/40]  Loss: 

RuntimeError: Parent directory AuraVision/models does not exist.

## Step 4 — Evaluate on Test Set

In [5]:
from train_model import build_test_loader, run_inference, TEST_DIR

test_loader, test_dataset = build_test_loader(TEST_DIR, class_to_idx)
print(f'Test images: {len(test_dataset)}\n')
run_inference(model, test_loader, test_dataset)

Test images: 152


Image                               True         Predicted    Confidence
adults_in_gym_11.jpg                Adults       Adults          98.30%  ✓
  ↳ All scores: {'Adults': '98.30%', 'Children': '1.64%', 'Seniors': '0.06%'}
adults_in_gym_2.jpg                 Adults       Adults          92.32%  ✓
  ↳ All scores: {'Adults': '92.32%', 'Children': '2.27%', 'Seniors': '5.41%'}
adults_in_gym_20.jpg                Adults       Adults          98.83%  ✓
  ↳ All scores: {'Adults': '98.83%', 'Children': '0.98%', 'Seniors': '0.19%'}
adults_in_gym_22.jpg                Adults       Adults          63.70%  ✓
  ↳ All scores: {'Adults': '63.70%', 'Children': '35.99%', 'Seniors': '0.32%'}
adults_in_gym_23.jpg                Children     Children        92.44%  ✓
  ↳ All scores: {'Adults': '4.52%', 'Children': '92.44%', 'Seniors': '3.04%'}
adults_in_gym_29.jpg                Adults       Adults          97.02%  ✓
  ↳ All scores: {'Adults': '97.02%', 'Children': '2.83%', 'Seniors'

([0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  2,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  2,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  2,
  1,
  2,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
